# Module 1: Feature Engineering

## NovaMart AI Retail Intelligence Platform

### Overview

This notebook presents the **Feature Engineering** phase of the NovaMart AI Retail Intelligence Platform. Feature engineering transforms the cleaned and integrated retail data into meaningful business variables that improve data analysis and machine learning model performance.

Using the master dataset developed during the data acquisition and integration phase, new features are created to capture customer behaviour, purchasing patterns, delivery performance, payment characteristics, product popularity, seller performance, and temporal trends. These engineered features provide richer business insights and serve as the foundation for the subsequent exploratory data analysis and machine learning modules.

### Objectives

The primary objectives of this notebook are to:

- Generate meaningful business features from the integrated retail dataset.
- Create time-based features such as purchase year, month, quarter, weekday, hour, and weekend indicator.
- Develop delivery-related features including delivery duration, shipping time, delivery delay, and late delivery status.
- Construct customer-level features such as purchase frequency, average order value, total spending, and Customer Lifetime Value (CLV).
- Generate product-level features including product popularity and revenue contribution.
- Create seller performance indicators such as total sales, average review score, and delivery performance.
- Prepare model-ready features for regression, classification, clustering, recommendation systems, and anomaly detection.
- Produce a feature-engineered dataset that will support exploratory data analysis, predictive modelling, explainable AI, and dashboard development.

### Expected Outcomes

At the end of this notebook, we will have:

- A comprehensive feature-engineered retail dataset containing meaningful business variables.
- New temporal, customer, product, seller, payment, and delivery features derived from the master dataset.
- Target variables required for predictive modelling, including delivery delay indicators.
- A high-quality analytical dataset ready for exploratory data analysis and machine learning.
- A reusable feature-engineered dataset (`Retail_Featured_Dataset.csv`) that will serve as the input for all subsequent modules of the NovaMart AI Retail Intelligence Platform.

# Import Required Libraries

The following libraries are imported to support data manipulation, statistical analysis, and data visualization throughout this exploratory data analysis.

- **Pandas** and **NumPy** for data manipulation and numerical computations.
- **Matplotlib** and **Seaborn** for statistical visualizations.
- **Plotly Express** for interactive charts.
- **Warnings** to suppress unnecessary warning messages for cleaner notebook output.

In [17]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Plot style
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

%matplotlib inline

# Load the Master Dataset

The cleaned and integrated master dataset developed during the data preprocessing phase is loaded into the notebook. This dataset serves as the single source of truth for all exploratory analyses and subsequent machine learning tasks.

The date columns are parsed as datetime objects to facilitate temporal analysis.

In [18]:
retail = pd.read_csv(
    "Retail_Master_Dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

print("Dataset Loaded Successfully")
print(f"Rows    : {retail.shape[0]:,}")
print(f"Columns : {retail.shape[1]}")

Dataset Loaded Successfully
Rows    : 113,314
Columns : 46


# Preview the Dataset

Before beginning the analysis, the first few records are displayed to verify that the dataset has been loaded correctly and to gain an initial understanding of its structure.

In [19]:
retail.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_value,payment_installments,payment_type,review_score,review_comment_title,review_comment_message,geolocation_zip_code_prefix,customer_lat,customer_lng,customer_geo_city,customer_geo_state,seller_lat,seller_lng,seller_geo_city,seller_geo_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,7.0,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,27277,volta redonda,SP,72.19,2.0,credit_card,5.0,NaN,"Perfeito, produto entregue antes do combinado.",28013.0,-21.763186,-41.310265,campos dos goytacazes,RJ,-22.497188,-44.127324,volta redonda,RJ
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,16.0,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,3471,sao paulo,SP,259.83,3.0,credit_card,4.0,NaN,NaN,15775.0,-20.222506,-50.898951,santa fe do sul,SP,-23.565754,-46.519097,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,7.0,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,37564,borda da mata,MG,216.87,5.0,credit_card,5.0,NaN,Chegou antes do prazo previsto e o produto surpreendeu pela qualidade. Muito satisfatório.,35661.0,-19.869998,-44.593059,para de minas,MG,-22.262802,-46.170735,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,6.0,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,14403,franca,SP,25.78,2.0,credit_card,4.0,NaN,NaN,12952.0,-23.105968,-46.590277,atibaia,SP,-20.553651,-47.387145,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,25.0,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,87900,loanda,PR,218.04,3.0,credit_card,5.0,NaN,Gostei pois veio no prazo determinado .,13226.0,-23.243402,-46.827614,varzea paulista,SP,-22.929583,-53.135750,loanda,PR


# Dataset Dimensions

Understanding the size of the dataset provides an indication of the volume of information available for analysis.

The dataset dimensions are reported as:

- Number of observations (rows)
- Number of variables (columns)

In [20]:
print("Rows :", retail.shape[0])
print("Columns :", retail.shape[1])

Rows : 113314
Columns : 46


# Data Structure

This section examines the structure of the dataset by displaying:

- Variable names
- Data types
- Number of non-null observations
- Memory usage

Understanding data types is essential before performing statistical analysis and visualization.

In [21]:
retail.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 113314 entries, 0 to 113313
Data columns (total 46 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       113314 non-null  object        
 1   order_item_id                  113314 non-null  int64         
 2   product_id                     113314 non-null  object        
 3   seller_id                      113314 non-null  object        
 4   shipping_limit_date            113314 non-null  object        
 5   price                          113314 non-null  float64       
 6   freight_value                  113314 non-null  float64       
 7   customer_id                    113314 non-null  object        
 8   order_status                   113314 non-null  object        
 9   order_purchase_timestamp       113314 non-null  datetime64[ns]
 10  order_approved_at              113299 non-null  datetime64[ns]
 11  

# Data Types Summary

The following table summarizes the number of variables belonging to each data type.

In [22]:
retail.dtypes.value_counts()

object            20
float64           18
datetime64[ns]     5
int64              3
dtype: int64

# Descriptive Statistics

Descriptive statistics provide an overview of the numerical variables, including:

- Mean
- Standard deviation
- Minimum
- Quartiles
- Maximum

These statistics help identify unusual values and understand the distribution of numerical attributes.

In [23]:
retail.describe(include="all").T

,count,unique,top,freq,first,last,mean,std,min,25%,50%,75%,max
order_id,113314,98666,5a3b1c29a49756e75f1ef513383c0c12,22,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_item_id,113314.0,NaN,NaN,NaN,NaT,NaT,1.198528,0.707016,1.0,1.0,1.0,1.0,21.0
product_id,113314,32951,aca2eb7d00ea1a7b8ebd4e68314663af,527,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_id,113314,3095,6560211a19b47992c3666cc44a7e94c0,2039,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
shipping_limit_date,113314,93318,2017-10-24 13:06:21,22,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,113314.0,NaN,NaN,NaN,NaT,NaT,120.478701,183.279678,0.85,39.9,74.9,134.9,6735.0
freight_value,113314.0,NaN,NaN,NaN,NaT,NaT,19.979428,15.783227,0.0,13.08,16.26,21.15,409.68
customer_id,113314,98666,be1c4e52bb71e0c54b11a26b8e8d59f2,22,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_status,113314,7,delivered,110840,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_purchase_timestamp,113314,98112,2017-10-17 13:06:29,22,2016-09-04 21:15:19,2018-09-03 09:06:57,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Descriptive Statistics for Categorical Variables

Categorical variables are summarized separately to identify:

- Number of unique categories
- Most frequent category
- Frequency of occurrence

In [24]:
retail.describe(include="object").T

,count,unique,top,freq
order_id,113314,98666,5a3b1c29a49756e75f1ef513383c0c12,22
product_id,113314,32951,aca2eb7d00ea1a7b8ebd4e68314663af,527
seller_id,113314,3095,6560211a19b47992c3666cc44a7e94c0,2039
shipping_limit_date,113314,93318,2017-10-24 13:06:21,22
customer_id,113314,98666,be1c4e52bb71e0c54b11a26b8e8d59f2,22
order_status,113314,7,delivered,110840
customer_unique_id,113314,95420,d97b3cfb22b0d6b25ac9ed4e9c2d481b,24
customer_city,113314,4110,sao paulo,17933
customer_state,113314,27,SP,47720
product_category_name,113314,74,cama_mesa_banho,11270


# Missing Value Analysis

Although the dataset has undergone preprocessing, a final assessment of missing values is performed to identify variables with remaining missing observations.

Some missing values are expected because they represent valid business scenarios, such as undelivered orders or customers who chose not to submit reviews.

In [25]:
missing = pd.DataFrame({
    "Missing Values": retail.isnull().sum(),
    "Percentage (%)": round(retail.isnull().mean()*100,2)
})

missing = missing[missing["Missing Values"]>0]

missing.sort_values("Missing Values",ascending=False)

,Missing Values,Percentage (%)
review_comment_title,99880,88.14
review_comment_message,65672,57.96
order_delivered_customer_date,2475,2.18
delivery_days,2475,2.18
product_category_name_english,1636,1.44
order_delivered_carrier_date,1203,1.06
review_score,942,0.83
customer_geo_state,307,0.27
geolocation_zip_code_prefix,307,0.27
customer_lat,307,0.27


# Duplicate Records

Duplicate records can negatively affect statistical analysis and machine learning models. The following check confirms whether duplicate observations remain in the integrated dataset.

In [26]:
duplicates = retail.duplicated().sum()

print(f"Duplicate Rows : {duplicates}")

Duplicate Rows : 265


In [27]:
# Step 1: Verify they are true duplicates

duplicate_rows = retail[retail.duplicated(keep=False)]

print(f"Number of duplicate rows: {duplicate_rows.shape[0]}")

duplicate_rows.head()

Number of duplicate rows: 527


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_value,payment_installments,payment_type,review_score,review_comment_title,review_comment_message,geolocation_zip_code_prefix,customer_lat,customer_lng,customer_geo_city,customer_geo_state,seller_lat,seller_lng,seller_geo_city,seller_geo_state
633,0176a6846bcb3b0d3aa3116a9a768597,1,19421075ae0b585f2dc13ff149e2119d,4c2b230173bb36f9b240f2b8ac11786e,2018-01-03 04:32:07,45.90,7.78,aaad67429ea7d339bdf5f71ae3ef74f6,delivered,2017-12-25 10:40:52,2017-12-27 04:32:07,2017-12-28 19:57:15,2017-12-29 21:22:35,2018-01-15,4.0,13613cae6e3bfea8327a916246c96cc9,5108,sao paulo,SP,esporte_lazer,32.0,1797.0,1.0,150.0,16.0,16.0,11.0,sports_leisure,3933,sao paulo,SP,53.68,1.0,boleto,5.0,NaN,NaN,5108.0,-23.509319,-46.758043,sao paulo,SP,-23.585019,-46.498841,sao paulo,SP
634,0176a6846bcb3b0d3aa3116a9a768597,1,19421075ae0b585f2dc13ff149e2119d,4c2b230173bb36f9b240f2b8ac11786e,2018-01-03 04:32:07,45.90,7.78,aaad67429ea7d339bdf5f71ae3ef74f6,delivered,2017-12-25 10:40:52,2017-12-27 04:32:07,2017-12-28 19:57:15,2017-12-29 21:22:35,2018-01-15,4.0,13613cae6e3bfea8327a916246c96cc9,5108,sao paulo,SP,esporte_lazer,32.0,1797.0,1.0,150.0,16.0,16.0,11.0,sports_leisure,3933,sao paulo,SP,53.68,1.0,boleto,5.0,NaN,NaN,5108.0,-23.509319,-46.758043,sao paulo,SP,-23.585019,-46.498841,sao paulo,SP
1447,03515a836bb855b03f7df9dee520a8fc,1,2b4609f8948be18874494203496bc318,cc419e0650a3c5ba77189a1882b7556a,2018-01-29 13:36:01,89.99,16.39,153e6d880db3cac30ecfeb13e0047841,delivered,2018-01-19 13:24:38,2018-01-19 13:36:01,2018-01-19 21:47:56,2018-02-05 14:33:00,2018-03-01,17.0,159306b86e782097be9a4e29e3d294d3,95540,palmares do sul,RS,beleza_saude,59.0,492.0,3.0,250.0,22.0,10.0,18.0,health_beauty,9015,santo andre,SP,106.38,10.0,credit_card,5.0,NaN,NaN,95540.0,-30.272859,-50.493819,palmares do sul,RS,-23.659354,-46.523359,santo andre,SP
1448,03515a836bb855b03f7df9dee520a8fc,1,2b4609f8948be18874494203496bc318,cc419e0650a3c5ba77189a1882b7556a,2018-01-29 13:36:01,89.99,16.39,153e6d880db3cac30ecfeb13e0047841,delivered,2018-01-19 13:24:38,2018-01-19 13:36:01,2018-01-19 21:47:56,2018-02-05 14:33:00,2018-03-01,17.0,159306b86e782097be9a4e29e3d294d3,95540,palmares do sul,RS,beleza_saude,59.0,492.0,3.0,250.0,22.0,10.0,18.0,health_beauty,9015,santo andre,SP,106.38,10.0,credit_card,5.0,NaN,NaN,95540.0,-30.272859,-50.493819,palmares do sul,RS,-23.659354,-46.523359,santo andre,SP
2334,05452881e2846549b81d39249bb66ad7,1,283dc451ad3918badb976d56ff887289,da8622b14eb17ae2831f4ac5b9dab84a,2018-02-11 22:35:33,99.90,12.40,1ff911e9682875f08a0f44bad0807455,delivered,2018-02-04 22:04:37,2018-02-05 22:35:33,2018-02-06 23:53:04,2018-02-07 22:11:23,2018-02-26,3.0,744fb2d41ededa16d3f703b7065b234c,2215,sao paulo,SP,cama_mesa_banho,58.0,188.0,1.0,1650.0,44.0,2.0,35.0,bed_bath_table,13405,piracicaba,SP,224.60,8.0,credit_card,5.0,NaN,NaN,2215.0,-23.490941,-46.583480,sao paulo,SP,-22.708485,-47.664918,piracicaba,SP


### They are duplicate records introduced during the integration process (most likely during one of the merges).

In [28]:
retail.iloc[633].equals(retail.iloc[634])

True

In [29]:
#Step 2: Remove them

#Since they are exact duplicates, remove them safely:

print("Shape before removing duplicates:")
print(retail.shape)

retail = retail.drop_duplicates()

print("\nShape after removing duplicates:")
print(retail.shape)

print("\nRemaining duplicates:")
print(retail.duplicated().sum())

Shape before removing duplicates:
(113314, 46)

Shape after removing duplicates:
(113049, 46)

Remaining duplicates:
0


# Logistics Feature Engineering

Efficient order fulfilment is a key performance indicator in e-commerce. This section creates logistics-related features that describe different stages of the delivery process.

The following features are generated:

- **Processing Time (`processing_days`)** – Time taken by the seller to prepare and hand over the order to the carrier.
- **Shipping Time (`shipping_days`)** – Time taken by the carrier to transport the order from the seller to the customer.
- **Delivery Time (`delivery_days`)** – Total time from order purchase to customer delivery.

These features will support exploratory data analysis, logistics performance evaluation, and delivery delay prediction models.

In [33]:
# =====================================================
# Logistics Feature Engineering
# =====================================================

# Seller processing time
retail["processing_days"] = (
    retail["order_delivered_carrier_date"]
    - retail["order_approved_at"]
).dt.days

# Carrier shipping time
retail["shipping_days"] = (
    retail["order_delivered_customer_date"]
    - retail["order_delivered_carrier_date"]
).dt.days

# Total customer waiting time
retail["delivery_days"] = (
    retail["order_delivered_customer_date"]
    - retail["order_purchase_timestamp"]
).dt.days

print("Logistics features created successfully.")

Logistics features created successfully.


## Verify the Engineered Features

The first few records of the newly created logistics features are displayed to ensure that they have been generated correctly.

In [34]:
retail[
    [
        "processing_days",
        "shipping_days",
        "delivery_days"
    ]
].head()

,processing_days,shipping_days,delivery_days
0,6.0,1.0,7.0
1,8.0,8.0,16.0
2,1.0,6.0,7.0
3,2.0,4.0,6.0
4,11.0,13.0,25.0


## Summary Statistics

The descriptive statistics below provide an overview of the newly engineered logistics features.

In [35]:
#Missing Values
retail[
    [
        "processing_days",
        "shipping_days",
        "delivery_days"
    ]
].isnull().sum()

processing_days    1216
shipping_days      2471
delivery_days      2470
dtype: int64

### Missing Values in Logistics Features

The logistics features (`processing_days`, `shipping_days`, and `delivery_days`) contain missing values because they are calculated from delivery-related timestamps.

These missing values correspond to orders that were **not successfully delivered**, including orders that were cancelled, still being processed, shipped but not yet delivered, or marked as unavailable.

Since these missing values represent valid business events rather than data quality issues, they were **retained** and not imputed. For analyses of logistics performance, only orders with a status of **`delivered`** are considered.

In [36]:
# Delivery delay relative to estimated delivery date
retail["delivery_delay_days"] = (
    retail["order_delivered_customer_date"]
    - retail["order_estimated_delivery_date"]
).dt.days

# Late delivery indicator
retail["late_delivery"] = (
    retail["delivery_delay_days"] > 0
).astype(int)

## Duplicate Record Analysis

Following data integration, the master dataset was examined for duplicate observations.

A total of **265 duplicate records** were identified. Detailed inspection confirmed that these records were exact duplicates introduced during the data integration process rather than representing genuine business transactions.

The duplicate records were removed using the `drop_duplicates()` function to ensure the integrity of the analytical dataset.

The final master dataset contains **113,049 observations**, **46 variables**, and **no duplicate records**, making it suitable for exploratory analysis and subsequent machine learning tasks.

In [37]:
print("="*60)
print("Duplicate Record Analysis")
print("="*60)

print(f"Total Rows             : {113049:,}")
print(f"Total Columns          : {retail.shape[1]}")
print(f"Remaining Duplicates   : {retail.duplicated().sum()}")

Duplicate Record Analysis
Total Rows             : 113,049
Total Columns          : 50
Remaining Duplicates   : 0


## Save the final master dataset

In [38]:
#Save the final master dataset
retail.to_csv(
    "Retail_Featured_Dataset.csv",
    index=False
)

print("✓ Clean master dataset saved successfully.")

✓ Clean master dataset saved successfully.
